In [ ]:
#Q)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df=pd.read_csv("./natural_gas_data/natural-gas_zip/archive/daily.csv")
print("Data:")
print(df)
print("Length of Data:",len(df))

df=df.dropna()
print("Length of Data after dropping null:",len(df))

y=df["Price"].values
x=np.arange(1,len(y))

epochs = 1500
print("\nLength of y: ",len(y))

minm=y.min()
maxm=y.max()
print(f"Minimum val: {minm},Maximum value: {maxm}")

y=(y-minm)/(maxm-minm) #normalize the ip range

Sequence_length=10

X=[]
Y=[]
for i in range(0,5900):
    list1=[]
    for j in range(i,i+Sequence_length):
        list1.append(y[j])
    X.append(list1)
    Y.append(y[j+1])

X=np.array(X)
Y=np.array(Y)

x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.10,random_state=42,
                                               shuffle=False,stratify=None)

class NGTimeSeries(Dataset):
    def __init__(self,x,y):
        self.x=torch.tensor(x,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
        self.len=x.shape[0]
    def __getitem__(self, idx):
        return self.x[idx],self.y[idx]
    def __len__(self):
        return self.len

dataset=NGTimeSeries(x_train,y_train)

train_loader=DataLoader(dataset,batch_size=256,shuffle=True)

class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn=nn.RNN(input_size=1,hidden_size=5,num_layers=1,batch_first=True)
        self.fc1=nn.Linear(in_features=5,out_features=1)
    def forward(self, x):
        output, _status = self.rnn(x)        
        output = output[:, -1, :]            
        output = self.fc1(torch.relu(output))
        return output

model = RNNModel()

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
epochs = 1500

for i in range(epochs):
    for j, data in enumerate(train_loader):
        y_pred = model(data[:][0].view(-1, Sequence_length, 1)).reshape(-1)
        loss = criterion(y_pred, data[:][1])
        loss.backward()
        optimizer.step()
    if i % 50 == 0:
        print(i, "th iteration : ", loss)

test_set = NGTimeSeries(x_test,y_test)
test_pred = model(test_set[:][0].view(-1,10,1)).view(-1)
plt.plot(test_pred.detach().numpy(),label='predicted')
plt.plot(test_set[:][1].view(-1),label='original')
plt.legend()
plt.show()

y = y * (maxm - minm) + minm
y_pred = test_pred.detach().numpy() * (maxm - minm) + minm
plt.plot(y)
plt.plot(range(len(y)-len(y_pred), len(y)), y_pred)
plt.show()



Data:
            Date  Price
0     1997-01-07   3.82
1     1997-01-08   3.80
2     1997-01-09   3.61
3     1997-01-10   3.92
4     1997-01-13   4.00
...          ...    ...
5948  2020-08-26   2.52
5949  2020-08-27   2.52
5950  2020-08-28   2.46
5951  2020-08-31   2.30
5952  2020-09-01   2.22

[5953 rows x 2 columns]
Length of Data: 5953
Length of Data after dropping null: 5952

Length of y:  5952
Minimum val: 1.05,Maximum value: 18.48
0 th iteration :  tensor(0.0343, grad_fn=<MseLossBackward0>)
50 th iteration :  tensor(0.0218, grad_fn=<MseLossBackward0>)
100 th iteration :  tensor(0.0191, grad_fn=<MseLossBackward0>)
150 th iteration :  tensor(0.0197, grad_fn=<MseLossBackward0>)
200 th iteration :  tensor(0.0172, grad_fn=<MseLossBackward0>)
250 th iteration :  tensor(0.0200, grad_fn=<MseLossBackward0>)


In [ ]:
import glob
import os
import string
import unicodedata
import random
import math
import torch
import torch.nn as nn

DATA_PATH = "data/names"
HIDDEN_SIZE = 128
LEARNING_RATE = 0.005
N_ITERS = 100000
PRINT_EVERY = 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_letters = string.ascii_letters 
n_letters = len(all_letters)

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

def load_data(path):
    category_lines = {}
    all_categories = []

    for filename in glob.glob(path + '/*.txt'):
        category = os.path.splitext(os.path.basename(filename))[0]
        all_categories.append(category)

        with open(filename, encoding='utf-8') as f:
            lines = f.read().strip().split('\n')
            lines = [unicodeToAscii(line) for line in lines]
            category_lines[category] = lines

    return category_lines, all_categories

category_lines, all_categories = load_data(DATA_PATH)
n_categories = len(all_categories)

print("Loaded languages:", n_categories)

def letterToIndex(letter):
    return all_letters.find(letter)

def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters, device=DEVICE)
    for i, letter in enumerate(line):
        idx = letterToIndex(letter)
        if idx != -1:
            tensor[i][0][idx] = 1
    return tensor


class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.rnn=nn.RNN(input_size,hidden_size)
        self.fc=nn.Linear(hidden_size,output_size)
        
    def forward(self, input, hidden):
        output,hidden=self.rnn(input,hidden)
        output=self.fc(output[-1])

        return output, hidden

    def initHidden(self):
        return torch.zeros(1,1, self.hidden_size, device=DEVICE)

def randomTrainingExample():
    category = random.choice(all_categories)
    line = random.choice(category_lines[category])

    category_tensor = torch.tensor(
        [all_categories.index(category)],
        dtype=torch.long,
        device=DEVICE
    )

    line_tensor = lineToTensor(line)
    return category, line, category_tensor, line_tensor

rnn = RNN(n_letters, HIDDEN_SIZE, n_categories).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(rnn.parameters(), lr=LEARNING_RATE)

def train(category_tensor, line_tensor):
    hidden = rnn.initHidden()
    optimizer.zero_grad()

    for i in range(line_tensor.size(0)):
        output, hidden = rnn(line_tensor[i].unsqueeze(0), hidden)

    loss = criterion(output, category_tensor)
    loss.backward()

    optimizer.step()

    return output, loss.item()

for i in range(1, N_ITERS + 1):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    output, loss = train(category_tensor, line_tensor)

    if i % PRINT_EVERY == 0:
        guess_i = torch.argmax(output).item()
        guess = all_categories[guess_i]

        correct = " Correct " if guess == category else f"Wrong- Actual: ({category})"

        print(f"{i} |  Loss: {loss:.4f} | {line} → {guess} {correct}")

torch.save(rnn.state_dict(), "rnn_name_classifier.pth")

print("\nModel saved!")

def predict(input_line, n_predictions=3):
    with torch.no_grad():
        line_tensor = lineToTensor(input_line)
        hidden = rnn.initHidden()

        for i in range(line_tensor.size(0)):
            output, hidden = rnn(line_tensor[i].unsqueeze(0), hidden)

        probs = torch.softmax(output, dim=1)
        topv, topi = probs.topk(n_predictions, 1, True)

        predictions = []
        for i in range(n_predictions):
            value = topv[0][i].item()
            category_index = topi[0][i].item()
            predictions.append((all_categories[category_index], value))

        return predictions


print("\nPredictions:")
test_names = ["Schmidt", "Garcia", "Ivanov", "Kim"]
pred=["German","Spanish","Russian","Korean"]

for i in range(len(test_names)):
    name=test_names[i]
    preds = predict(name)
    print(f"\n{name}:")
    max_sc=float('-inf')
    max_lang=None
    for lang, score in preds:
        print(f"  {lang} ({score:.4f})")
        if (score>max_sc):
            max_sc=score
            max_lang=lang
    print("Predicted Language: ", max_lang)
    print("Actual Language: ", pred[i])

Loaded languages: 18
5000 |  Loss: 3.0448 | Tasse → Arabic Wrong- Actual: (French)
10000 |  Loss: 1.8384 | Marquez → Spanish  Correct 
15000 |  Loss: 2.2296 | Zawisza → Japanese Wrong- Actual: (Polish)
20000 |  Loss: 0.7638 | Hao → Chinese  Correct 
25000 |  Loss: 0.1174 | Antonopoulos → Greek  Correct 
30000 |  Loss: 1.3236 | Deniel → Irish Wrong- Actual: (French)
35000 |  Loss: 0.5909 | Xydis → Greek  Correct 
40000 |  Loss: 2.0391 | Masi → Japanese Wrong- Actual: (Italian)
45000 |  Loss: 3.7823 | Oorschot → Scottish Wrong- Actual: (Dutch)
50000 |  Loss: 2.0503 | Daramitz → Spanish Wrong- Actual: (French)
55000 |  Loss: 1.0549 | Peusen → Dutch  Correct 
60000 |  Loss: 0.2779 | Parisi → Italian  Correct 
65000 |  Loss: 3.5897 | Tracey → French Wrong- Actual: (Irish)
70000 |  Loss: 0.4011 | Corti → Italian  Correct 
75000 |  Loss: 0.8192 | Hakimi → Japanese Wrong- Actual: (Arabic)
80000 |  Loss: 0.0901 | Diakogeorgiou → Greek  Correct 
85000 |  Loss: 1.6214 | Kim → Korean Wrong- Actual

In [ ]:
#Q3
import torch
import torch.nn as nn
import random
import string

all_characters = string.printable
n_chars = len(all_characters)

HIDDEN_SIZE = 128
LEARNING_RATE = 0.005
N_ITERS = 20000
PRINT_EVERY = 2000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text = """hello world
deep learning"""

def charToIndex(c):
    return all_characters.find(c)

def charToTensor(c):
    tensor = torch.zeros(1, n_chars, device=DEVICE)
    tensor[0][charToIndex(c)] = 1
    return tensor

def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_chars, device=DEVICE)
    for i, c in enumerate(line):
        tensor[i][0][charToIndex(c)] = 1
    return tensor

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.rnn = nn.RNN(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):
        output, hidden = self.rnn(input, hidden)
        output = self.fc(output[-1])   # last timestep
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=DEVICE)

def randomTrainingExample():
    start_index = random.randint(0, len(text) - 2)
    end_index = start_index + random.randint(5, 15)

    chunk = text[start_index:end_index]

    input_seq = chunk[:-1]
    target_seq = chunk[1:]

    input_tensor = lineToTensor(input_seq)
    target_tensor = torch.tensor(
        [charToIndex(c) for c in target_seq],
        dtype=torch.long,
        device=DEVICE
    )

    return input_tensor, target_tensor

rnn = RNN(n_chars, HIDDEN_SIZE, n_chars).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(rnn.parameters(), lr=LEARNING_RATE)

def train(input_tensor, target_tensor):
    hidden = rnn.initHidden()
    optimizer.zero_grad()

    loss = 0

    for i in range(input_tensor.size(0)):
        output, hidden = rnn(input_tensor[i].unsqueeze(0), hidden)
        loss += criterion(output, target_tensor[i].unsqueeze(0))

    loss.backward()

    optimizer.step()

    return loss.item() / input_tensor.size(0)

for i in range(1, N_ITERS + 1):
    input_tensor, target_tensor = randomTrainingExample()
    loss = train(input_tensor, target_tensor)

    if i % PRINT_EVERY == 0:
        print(f"Iter {i} | Loss: {loss:.4f}")

def generate(start_str="he", predict_len=100):
    with torch.no_grad():
        input_tensor = lineToTensor(start_str)
        hidden = rnn.initHidden()

        # feed initial string
        for i in range(len(start_str) - 1):
            _, hidden = rnn(input_tensor[i].unsqueeze(0), hidden)

        last_char = input_tensor[-1]
        output_str = start_str

        for _ in range(predict_len):
            output, hidden = rnn(last_char.unsqueeze(0), hidden)

            probs = torch.softmax(output, dim=1)
            topi = torch.multinomial(probs, 1).item()

            predicted_char = all_characters[topi]
            output_str += predicted_char

            last_char = charToTensor(predicted_char)

        return output_str

print("\nGenerated Text:\n")
print(generate("deep ", 8))
print(generate("hello ", 5))
print(generate("world ", 5))

Iter 2000 | Loss: 0.2521


KeyboardInterrupt: 

# EASIER VERSION

import torch
import torch.nn as nn
import os
import string
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_path = "data/data/names"

all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)

languages = []
lang_to_idx = {}

def letter_to_index(letter):
    return all_letters.find(letter)

def name_to_tensor(name):
    tensor = torch.zeros(len(name), n_letters)
    for i, letter in enumerate(name):
        idx = letter_to_index(letter)
        if idx != -1:
            tensor[i][idx] = 1
    return tensor

x = []
y = []

for idx, filename in enumerate(os.listdir(data_path)):
    if filename.endswith(".txt"):
        lang = filename.split(".")[0]
        languages.append(lang)
        lang_to_idx[lang] = idx

        with open(os.path.join(data_path, filename), encoding="utf-8") as f:
            names = f.read().strip().split("\n")
            for name in names:
                x.append(name)
                y.append(idx)

class mydataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return name_to_tensor(self.x[i]), torch.tensor(self.y[i], dtype=torch.long)

train_ds = mydataset(x, y)
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True)

class rnn_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=n_letters,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(64, len(languages))

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

model = rnn_model().to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

epochs = 15

for epoch in range(epochs):
    total_loss = 0
    for x_batch, y_batch in train_dl:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss = {total_loss:.4f}")

def predict(name):
    with torch.no_grad():
        tensor = name_to_tensor(name).unsqueeze(0).to(device)
        out = model(tensor)
        pred_idx = torch.argmax(out).item()
        return languages[pred_idx]

print(predict("Aleshire"))
print(predict("Antonopoulos"))
print(predict("Abbott"))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text = "hello"

chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

def encode_one_hot(seq):
    tensor = torch.zeros(len(seq), vocab_size)
    for i, ch in enumerate(seq):
        tensor[i][char_to_idx[ch]] = 1
    return tensor

sequence_length = 4
x = []
y = []

for i in range(len(text) - sequence_length):
    x.append(encode_one_hot(text[i:i+sequence_length]))
    y.append(char_to_idx[text[i+sequence_length]])

x = torch.stack(x)
y = torch.tensor(y, dtype=torch.long)

class CharDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_ds = CharDataset(x, y)
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True)

class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=vocab_size,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

model = RNNModel().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 300

for epoch in range(epochs):
    for x_batch, y_batch in train_dl:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def predict_next(seq):
    model.eval()
    with torch.no_grad():
        x = encode_one_hot(seq).unsqueeze(0).to(device)
        out = model(x)
        pred_idx = torch.argmax(out).item()
        return idx_to_char[pred_idx]

print(predict_next("hell"))

# PRACTICE

In [ ]:
import glob
import os
import string
import unicodedata
import random
import math
import torch
import torch.nn as nn

DATA_PATH = "data/names"
HIDDEN_SIZE = 128
LEARNING_RATE = 0.005
N_ITERS = 100000
PRINT_EVERY = 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_letters = string.ascii_letters 
n_letters = len(all_letters)

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

def load_data(path):
    category_lines = {}
    all_categories = []

    for filename in glob.glob(path + '/*.txt'):
        category = os.path.splitext(os.path.basename(filename))[0]
        all_categories.append(category)

        with open(filename, encoding='utf-8') as f:
            lines = f.read().strip().split('\n')
            lines = [unicodeToAscii(line) for line in lines]
            category_lines[category] = lines

    return category_lines, all_categories

category_lines, all_categories = load_data(DATA_PATH)
n_categories = len(all_categories)

print("Loaded languages:", n_categories)

def letterToIndex(letter):
    return all_letters.find(letter)

def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters, device=DEVICE)
    for i, letter in enumerate(line):
        idx = letterToIndex(letter)
        if idx != -1:
            tensor[i][0][idx] = 1
    return tensor

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.i2o = nn.Linear(input_size + hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        combined = torch.cat((input, hidden), 1)

        hidden = self.i2h(combined)
        output = self.i2o(combined)
        output = self.softmax(output)

        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size, device=DEVICE)

def randomTrainingExample():
    category = random.choice(all_categories)
    line = random.choice(category_lines[category])

    category_tensor = torch.tensor(
        [all_categories.index(category)],
        dtype=torch.long,
        device=DEVICE
    )

    line_tensor = lineToTensor(line)
    return category, line, category_tensor, line_tensor

rnn = RNN(n_letters, HIDDEN_SIZE, n_categories).to(DEVICE)
criterion = nn.NLLLoss()

def train(category_tensor, line_tensor):
    hidden = rnn.initHidden()
    rnn.zero_grad()

    for i in range(line_tensor.size(0)):
        output, hidden = rnn(line_tensor[i], hidden)

    loss = criterion(output, category_tensor)
    loss.backward()

    # manual SGD
    for p in rnn.parameters():
        p.data.add_(p.grad.data, alpha=-LEARNING_RATE)

    return output, loss.item()


for i in range(1, N_ITERS + 1):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    output, loss = train(category_tensor, line_tensor)

    if i % PRINT_EVERY == 0:
        guess_i = torch.argmax(output).item()
        guess = all_categories[guess_i]

        correct = " Correct " if guess == category else f"Wrong- Actual: ({category})"

        print(f"{i} |  Loss: {loss:.4f} | {line} → {guess} {correct}")

torch.save(rnn.state_dict(), "rnn_name_classifier.pth")

print("\nModel saved!")

def predict(input_line, n_predictions=3):
    with torch.no_grad():
        line_tensor = lineToTensor(input_line)
        hidden = rnn.initHidden()

        for i in range(line_tensor.size(0)):
            output, hidden = rnn(line_tensor[i], hidden)

        topv, topi = output.topk(n_predictions, 1, True)

        predictions = []
        for i in range(n_predictions):
            value = topv[0][i].item()
            category_index = topi[0][i].item()
            predictions.append((all_categories[category_index], value))

        return predictions


print("\nPredictions:")
test_names = ["Schmidt", "Garcia", "Ivanov", "Kim"]
pred=["German","Portugese","Russian","Korean"]

for i in range(len(test_names)):
    name=test_names[i]
    preds = predict(name)
    print(f"\n{name}:")
    max_sc=float('-inf')
    max_lang=None
    for lang, score in preds:
        print(f"  {lang} ({score:.4f})")
        if (score>max_sc):
            max_sc=score
            max_lang=lang
    print("Predicted Language: ", lang)
    print("Actual Language: ", pred[i])

In [ ]:
import torch
import torch.nn as nn
import random
import string

all_characters = string.printable
n_chars = len(all_characters)

HIDDEN_SIZE = 128
LEARNING_RATE = 0.005
N_ITERS = 20000
PRINT_EVERY = 2000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text = """hello world
deep learning"""

def charToIndex(c):
    return all_characters.find(c)

def charToTensor(c):
    tensor = torch.zeros(1, n_chars, device=DEVICE)
    tensor[0][charToIndex(c)] = 1
    return tensor

def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_chars, device=DEVICE)
    for i, c in enumerate(line):
        tensor[i][0][charToIndex(c)] = 1
    return tensor

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.i2o = nn.Linear(input_size + hidden_size, output_size)

    def forward(self, input, hidden):
        combined = torch.cat((input, hidden), 1)

        hidden = torch.tanh(self.i2h(combined))
        output = self.i2o(combined)

        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size, device=DEVICE)

def randomTrainingExample():
    start_index = random.randint(0, len(text) - 2)
    end_index = start_index + random.randint(5, 15)

    chunk = text[start_index:end_index]

    input_seq = chunk[:-1]
    target_seq = chunk[1:]

    input_tensor = lineToTensor(input_seq)
    target_tensor = torch.tensor(
        [charToIndex(c) for c in target_seq],
        dtype=torch.long,
        device=DEVICE
    )

    return input_tensor, target_tensor

rnn = RNN(n_chars, HIDDEN_SIZE, n_chars).to(DEVICE)
criterion = nn.CrossEntropyLoss()

def train(input_tensor, target_tensor):
    hidden = rnn.initHidden()
    rnn.zero_grad()

    loss = 0

    for i in range(input_tensor.size(0)):
        output, hidden = rnn(input_tensor[i], hidden)
        loss += criterion(output, target_tensor[i].unsqueeze(0))

    loss.backward()

    for p in rnn.parameters():
        p.data.add_(p.grad.data, alpha=-LEARNING_RATE)

    return loss.item() / input_tensor.size(0)

for i in range(1, N_ITERS + 1):
    input_tensor, target_tensor = randomTrainingExample()
    loss = train(input_tensor, target_tensor)

    if i % PRINT_EVERY == 0:
        print(f"Iter {i} | Loss: {loss:.4f}")

def generate(start_str="he", predict_len=100):
    with torch.no_grad():
        input_tensor = lineToTensor(start_str)
        hidden = rnn.initHidden()

        # feed initial string
        for i in range(len(start_str) - 1):
            _, hidden = rnn(input_tensor[i], hidden)

        last_char = input_tensor[-1]
        output_str = start_str

        for _ in range(predict_len):
            output, hidden = rnn(last_char, hidden)

            probs = torch.softmax(output, dim=1)
            topi = torch.multinomial(probs, 1)[0]

            predicted_char = all_characters[topi]
            output_str += predicted_char

            last_char = charToTensor(predicted_char)

        return output_str

print("\nGenerated Text:\n")
print(generate("deep ", 8))
print(generate("hello ", 5))
print(generate("world ", 5))